# Experiment 1 — mT5-small Baseline Fine-tuning

**Approach:** Fine-tune `google/mt5-small` on the full training dataset (~28k examples) for 3 epochs with learning rate 1e-4. No prompt engineering, no language tag — raw question as input.

**Zindi Score:** 0.198643

**Key Config:**
- Model: `google/mt5-small`
- Data: Full training set (~28,323 examples)
- Epochs: 3
- Learning rate: 1e-4
- Batch size: 2 + gradient accumulation steps: 8
- No language tag in prompt

## 1 — Install and Import Packages

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q scikit-learn pandas numpy rouge-score
!pip install -q transformers sentencepiece accelerate torch datasets

print('✅ Packages installed')

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

warnings.filterwarnings('ignore')
SEED = 42
print('✅ Imports done')

## 2 — Set File Paths

In [ ]:
DATA_DIR = Path('/content')

TRAIN_PATH      = DATA_DIR / 'Train.csv'
TEST_PATH       = DATA_DIR / 'Test.csv'
VAL_PATH        = DATA_DIR / 'Val.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'SampleSubmission.csv'

OUTPUT_TFIDF     = Path('/content/submission_tfidf_baseline.csv')
OUTPUT_LLM       = Path('/content/submission_llm_baseline.csv')
OUTPUT_FINETUNED = Path('/content/submission_exp1_mt5small_baseline.csv')

for path in [TRAIN_PATH, TEST_PATH, VAL_PATH, SAMPLE_SUB_PATH]:
    status = '✅' if path.exists() else '❌'
    print(f'{status} {path}')

## 3 — Load Data

In [ ]:
train             = pd.read_csv(TRAIN_PATH)
test              = pd.read_csv(TEST_PATH)
val               = pd.read_csv(VAL_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
print(f'Val shape   : {val.shape}')

print('\nLanguage distribution in training set:')
display(train['subset'].value_counts())

## 4 — Column Configuration

In [ ]:
ID_COL            = 'ID'
TEST_ID_COL       = 'ID'
QUESTION_COL      = 'input'
TEST_QUESTION_COL = 'input'
ANSWER_COL        = 'output'
LANG_COL          = 'subset'
TEST_LANG_COL     = 'subset'
GROUP_COL         = LANG_COL

SUBSET_TO_LANGUAGE = {
    'Eng': 'English',
    'Aka': 'Akan',
    'Lug': 'Luganda',
    'Swa': 'Swahili',
    'Amh': 'Amharic',
}

def subset_to_language_name(subset_code):
    prefix = str(subset_code).split('_')[0]
    return SUBSET_TO_LANGUAGE.get(prefix, subset_code)

print('✅ Columns configured')

## 5 — Text Cleaning

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip()

train[QUESTION_COL]      = train[QUESTION_COL].map(clean_text)
train[ANSWER_COL]        = train[ANSWER_COL].map(clean_text)
test[TEST_QUESTION_COL]  = test[TEST_QUESTION_COL].map(clean_text)
val[QUESTION_COL]        = val[QUESTION_COL].map(clean_text)
val[ANSWER_COL]          = val[ANSWER_COL].map(clean_text)

train = train[(train[QUESTION_COL] != '') & (train[ANSWER_COL] != '')].reset_index(drop=True)
print(f'Cleaned train shape : {train.shape}')

## 6 — ROUGE Evaluation

In [ ]:
try:
    from rouge_score import rouge_scorer

    class WhitespaceTokenizer:
        def tokenize(self, text):
            if text is None:
                return []
            return str(text).strip().split()

    _scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], tokenizer=WhitespaceTokenizer())

    def compute_rouge(predictions, references):
        r1_scores, rl_scores = [], []
        for pred, ref in zip(predictions, references):
            scores = _scorer.score(str(ref), str(pred))
            r1_scores.append(scores['rouge1'].fmeasure)
            rl_scores.append(scores['rougeL'].fmeasure)
        return {'rouge1_f1': float(np.mean(r1_scores)), 'rougeL_f1': float(np.mean(rl_scores))}

    def compute_rouge_by_language(predictions, references, languages):
        rows = []
        for lang in sorted(set(languages)):
            idx = [i for i, l in enumerate(languages) if l == lang]
            m = compute_rouge([predictions[i] for i in idx], [references[i] for i in idx])
            rows.append({'language': lang, 'rouge1_f1': m['rouge1_f1'], 'rougeL_f1': m['rougeL_f1']})
        return pd.DataFrame(rows).set_index('language')

    compute_rouge_available = True
    print('✅ ROUGE scoring available')
except Exception as e:
    compute_rouge_available = False
    print(f'⚠️ ROUGE unavailable: {e}')

## 7 — Model Configuration

In [ ]:
MODEL_NAME        = 'google/mt5-small'
MAX_INPUT_LENGTH  = 128
MAX_OUTPUT_LENGTH = 256
BATCH_SIZE_LLM    = 2
NUM_BEAMS         = 4

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_NAME}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 8 — Load Model

In [ ]:
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model_llm = model_llm.to(DEVICE)
model_llm.eval()
print(f'✅ {MODEL_NAME} loaded on {DEVICE}')
print(f'   Parameters : {sum(p.numel() for p in model_llm.parameters()) / 1e6:.0f}M')

## 9 — Prompt and Generation Functions

In [ ]:
import re

def build_prompt(question: str, language: str = None) -> str:
    # Exp 1: no language tag — raw question only
    return f'Answer this health question: {question}'

def generate_answers_batch(questions, languages=None, batch_size=BATCH_SIZE_LLM):
    if languages is None:
        languages = [None] * len(questions)
    all_answers = []
    n_batches = (len(questions) + batch_size - 1) // batch_size
    for batch_idx in range(n_batches):
        start = batch_idx * batch_size
        end   = min(start + batch_size, len(questions))
        prompts = [build_prompt(q, l) for q, l in zip(questions[start:end], languages[start:end])]
        inputs  = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_INPUT_LENGTH).to(DEVICE)
        with torch.no_grad():
            outputs = model_llm.generate(**inputs, max_new_tokens=MAX_OUTPUT_LENGTH, num_beams=NUM_BEAMS, no_repeat_ngram_size=3)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        cleaned = [re.sub(r'<extra_id_\d+>', '', ans).strip() for ans in decoded]
        all_answers.extend(cleaned)
        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == n_batches:
            print(f'  Batch {batch_idx + 1}/{n_batches} — {end}/{len(questions)} processed')
    return all_answers

print('✅ Generation functions defined')

## 10 — Fine-tuning Configuration

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset

FINETUNE_OUTPUT_DIR     = './mt5-small-exp1-baseline'
FINETUNE_EPOCHS         = 3
FINETUNE_BATCH_SIZE     = 2
FINETUNE_LEARNING_RATE  = 1e-4
FINETUNE_MAX_INPUT_LEN  = 128
FINETUNE_MAX_TARGET_LEN = 256
FINETUNE_VAL_SIZE       = 0.05
GRADIENT_ACCUM_STEPS    = 8

print(f'Fine-tuning config:')
print(f'  Model            : {MODEL_NAME}')
print(f'  Epochs           : {FINETUNE_EPOCHS}')
print(f'  Batch size       : {FINETUNE_BATCH_SIZE}')
print(f'  Learning rate    : {FINETUNE_LEARNING_RATE}')
print(f'  Max input tokens : {FINETUNE_MAX_INPUT_LEN}')
print(f'  Max target tokens: {FINETUNE_MAX_TARGET_LEN}')
print(f'  Val split        : {FINETUNE_VAL_SIZE:.0%}')
print(f'  Data             : Full training set (no sampling)')
print(f'  Output dir       : {FINETUNE_OUTPUT_DIR}')

## 11 — Prepare Dataset and Train

In [ ]:
def make_hf_dataset(df, question_col, answer_col, lang_col):
    records = []
    for _, row in df.iterrows():
        prompt = build_prompt(
            question=str(row[question_col]),
            language=str(row[lang_col]) if lang_col and lang_col in df.columns else None,
        )
        records.append({'prompt': prompt, 'answer': str(row[answer_col])})
    raw_ds = Dataset.from_list(records)

    def preprocess(examples):
        model_inputs = tokenizer(examples['prompt'], max_length=FINETUNE_MAX_INPUT_LEN, truncation=True, padding=False)
        labels = tokenizer(text_target=examples['answer'], max_length=FINETUNE_MAX_TARGET_LEN, truncation=True, padding=False)
        model_inputs['labels'] = [
            [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
            for seq in labels['input_ids']
        ]
        return model_inputs

    return raw_ds.map(preprocess, batched=True, remove_columns=['prompt', 'answer'])

train_df, val_ft_df = train_test_split(
    train,
    test_size=FINETUNE_VAL_SIZE,
    random_state=SEED,
    stratify=train[LANG_COL] if LANG_COL in train.columns else None,
)

# Exp 1: Full data — no sampling
print(f'Fine-tuning split — train: {len(train_df):,}  val: {len(val_ft_df):,}')

hf_train_ds = make_hf_dataset(train_df,  QUESTION_COL, ANSWER_COL, LANG_COL)
hf_val_ds   = make_hf_dataset(val_ft_df, QUESTION_COL, ANSWER_COL, LANG_COL)

print(f'HF train dataset : {hf_train_ds}')
print(f'HF val dataset   : {hf_val_ds}')

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_llm, label_pad_token_id=-100, pad_to_multiple_of=8)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = FINETUNE_OUTPUT_DIR,
    num_train_epochs            = FINETUNE_EPOCHS,
    per_device_train_batch_size = FINETUNE_BATCH_SIZE,
    per_device_eval_batch_size  = FINETUNE_BATCH_SIZE,
    gradient_accumulation_steps = GRADIENT_ACCUM_STEPS,
    learning_rate               = FINETUNE_LEARNING_RATE,
    predict_with_generate       = True,
    bf16                        = (DEVICE == 'cuda' and torch.cuda.is_bf16_supported()),
    fp16                        = (DEVICE == 'cuda' and not torch.cuda.is_bf16_supported()),
    eval_strategy               = 'epoch',
    save_strategy               = 'no',
    load_best_model_at_end      = False,
    logging_steps               = 100,
    generation_max_length       = FINETUNE_MAX_TARGET_LEN,
    report_to                   = 'none',
)

trainer = Seq2SeqTrainer(
    model            = model_llm,
    args             = training_args,
    train_dataset    = hf_train_ds,
    eval_dataset     = hf_val_ds,
    processing_class = tokenizer,
    data_collator    = data_collator,
)

print('Starting fine-tuning...')
print(f'  Training on {len(hf_train_ds):,} examples for {FINETUNE_EPOCHS} epoch(s)')
trainer.train()
print('\n✅ Fine-tuning complete')

## 12 — Generate Test Predictions and Save Submission

In [ ]:
print(f'Generating fine-tuned answers for {len(test):,} test questions...')
model_llm.eval()

test_questions_ft = test[TEST_QUESTION_COL].tolist()
test_languages_ft = test[TEST_LANG_COL].tolist() if TEST_LANG_COL else None
test_pred_finetuned = generate_answers_batch(test_questions_ft, test_languages_ft)

print(f'\n✅ Generated {len(test_pred_finetuned):,} answers')

# Validate on 500 val samples
if compute_rouge_available:
    val_sample_ft = val.sample(n=500, random_state=42)
    val_pred_ft   = generate_answers_batch(val_sample_ft[QUESTION_COL].tolist(), val_sample_ft[LANG_COL].tolist())
    metrics_ft    = compute_rouge(val_pred_ft, val_sample_ft[ANSWER_COL].tolist())
    print(f'\n📊 Fine-tuned — Validation ROUGE')
    print(f'   ROUGE-1 F1 : {metrics_ft["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics_ft["rougeL_f1"]:.4f}')
    lang_metrics = compute_rouge_by_language(val_pred_ft, val_sample_ft[ANSWER_COL].tolist(), val_sample_ft[LANG_COL].tolist())
    display(lang_metrics.round(4))

# Save submission
clean_preds = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in test_pred_finetuned]
sub = pd.DataFrame({'ID': test[TEST_ID_COL].values, 'TargetRLF1': clean_preds, 'TargetR1F1': clean_preds, 'TargetLLM': clean_preds})
sub.to_csv(OUTPUT_FINETUNED, index=False)
print(f'\n✅ Submission saved to: {OUTPUT_FINETUNED}')
print(f'   Shape : {sub.shape}')
display(sub.head())